# Generador de Dataset Sintético — Proyecto Segmentación Joyería de Lujo

**Proyecto de Titulación — Maestría en Inteligencia Artificial Aplicada**  
**Universidad de Las Américas (UDLA)**  
**Autores:** Jhon Sánchez · Vinicio Betancourt  

---

**Propósito:** Generar un dataset completamente sintético con las mismas 18 columnas que el dataset real, para que evaluadores puedan verificar el pipeline sin acceder a datos confidenciales.

**Instrucciones:** Ejecutar todas las celdas en orden: `Entorno de ejecución → Ejecutar todo`.
Al finalizar, descargar el CSV y subirlo a Google Drive en `/MyDrive/Proyecto_Swarovski/`.

In [1]:
import pandas as pd
import numpy as np
import hashlib
import random
from datetime import date, timedelta

SEED            = 2025
N_CLIENTES      = 3_000
PCT_DEVOLUCION  = 0.028
PCT_MULTICOMPRA = 0.22
FECHA_INICIO    = date(2023, 1, 1)
FECHA_FIN       = date(2025, 12, 31)
OUTPUT_FILE     = 'Dataset_Sintetico_Joyeria.csv'

np.random.seed(SEED)
random.seed(SEED)

print('Imports y constantes cargados.')
print(f'Clientes a generar : {N_CLIENTES:,}')
print(f'Periodo            : {FECHA_INICIO} a {FECHA_FIN}')

Imports y constantes cargados.
Clientes a generar : 3,000
Periodo            : 2023-01-01 a 2025-12-31


In [2]:
SECCIONES = {
    'COLLAR':0.28,'ARETES':0.27,'PULSERA':0.17,'ANILLO':0.14,
    'BOLIGRAFOS':0.06,'INDETERMINADO':0.04,'RELOJ Y SUS ACCESORIOS':0.02,
    'FAMILIA 70-80':0.01,'HOME':0.005,'LLAVEROS, MOBILE PHONE Y CHARM':0.005,
    'PRENDEDORES Y GEMELOS':0.0,
}
total_sec = sum(SECCIONES.values())
SEC_NAMES = list(SECCIONES.keys())
SEC_PROBS = [v/total_sec for v in SECCIONES.values()]

ZONAS = {
    'CENTRO COMERCIAL NORTE':0.30,'CENTRO COMERCIAL SUR':0.22,
    'MALL ESTE':0.20,'MALL OESTE':0.16,
    'ZONA CENTRO':0.07,'ZONA PERIFERICA':0.04,'NO REGISTRADO':0.01,
}
ZONA_NAMES = list(ZONAS.keys())
ZONA_PROBS = list(ZONAS.values())
BODEGA_MAP = {z:i+10 for i,z in enumerate(ZONA_NAMES)}

LINEAS      = {'JEWERLY':0.96,'LIVING':0.03,'ACCESORIOS':0.01}
LINEA_NAMES = list(LINEAS.keys())
LINEA_PROBS = list(LINEAS.values())

VENDEDORES = [f'V.{n}' for n in [
    'ANA','CARLOS','LUCIA','PEDRO','SOFIA','MARIO','ELENA',
    'DIEGO','CARMEN','JORGE','PAULA','RAUL','DIANA','IVAN',
    'ROSA','FELIX','VERA','HUGO','ALBA','TOMAS'
]]

MESES_ES = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',6:'Junio',
            7:'Julio',8:'Agosto',9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}

CIUDADES = ['NORTE','SUR','CENTRO','VALLES','OCCIDENTE',
            'ORIENTE','COSTA','SIERRA','AMAZONIA','INSULAR']

CODIGOS_PRODUCTO = {
    'COLLAR':list(range(5100000,5110000)),'ARETES':list(range(5200000,5210000)),
    'PULSERA':list(range(5300000,5310000)),'ANILLO':list(range(5400000,5410000)),
    'BOLIGRAFOS':list(range(5500000,5510000)),
}
DESCRIPCIONES = {
    'COLLAR':  ['NECKLACE CRYSTAL CLEAR M','PENDANT SWAN ROSE GOLD','NECKLACE CHARM SILVER L'],
    'ARETES':  ['EARRINGS STUD CRYSTAL S','EARRINGS HOOP GOLD M','PE STUDS ROSE CRYSTAL'],
    'PULSERA': ['BRACELET CHARM SILVER M','BRACELET TENNIS CRYSTAL L','BANGLE ROSE GOLD S'],
    'ANILLO':  ['RING CRYSTAL BAND M','RING SOLITAIRE CLEAR L','RING CHARM GOLD S'],
    'BOLIGRAFOS':['PEN CRYSTAL SILVER','PEN BALLPOINT GOLD','PEN LUXURY CLEAR'],
}
PRECIO_PARAMS = {
    'COLLAR':(85,480,165,55),'ARETES':(70,320,130,45),'PULSERA':(90,420,155,50),
    'ANILLO':(75,380,145,48),'BOLIGRAFOS':(60,250,110,35),'INDETERMINADO':(50,200,95,30),
    'RELOJ Y SUS ACCESORIOS':(120,550,210,80),'FAMILIA 70-80':(40,180,85,25),
    'HOME':(55,220,100,32),'LLAVEROS, MOBILE PHONE Y CHARM':(35,150,75,22),
    'PRENDEDORES Y GEMELOS':(45,170,90,28),
}
print('Catalogos definidos.')

Catalogos definidos.


In [3]:
def sha256(text):
    return hashlib.sha256(text.encode()).hexdigest()

def fecha_aleatoria(inicio, fin):
    delta = (fin - inicio).days
    return inicio + timedelta(days=random.randint(0, delta))

print('Funciones auxiliares listas.')

Funciones auxiliares listas.


In [4]:
print('Generando clientes sinteticos...')
clientes = []
for i in range(N_CLIENTES):
    ruc_raw  = f'SYN{i:07d}{random.randint(1000,9999)}'
    nom_raw  = f'CLIENTE_SINTETICO_{i:06d}'
    tel_raw  = f'09{random.randint(10000000,99999999)}'
    mail_raw = f'cliente{i}@example.com'
    ciudad   = random.choice(CIUDADES)
    clientes.append({
        'idx':i,
        'DOMICILIO':f'{ciudad} CALLE {random.randint(1,99)} N{random.randint(1,999)}',
        'RUC_HASH':sha256(ruc_raw),'NOM_HASH':sha256(nom_raw),
        'TEL_HASH':sha256(tel_raw),'MAIL_HASH':sha256(mail_raw),
    })

df_clientes = pd.DataFrame(clientes)
print(f'{len(df_clientes):,} clientes sinteticos generados.')

Generando clientes sinteticos...
3,000 clientes sinteticos generados.


In [5]:
print('Asignando transacciones...')
transacciones_base = []
for i in range(N_CLIENTES):
    transacciones_base.append((i, fecha_aleatoria(FECHA_INICIO, FECHA_FIN)))

n_multi   = int(N_CLIENTES * PCT_MULTICOMPRA)
idx_multi = np.random.choice(N_CLIENTES, n_multi, replace=False)
for idx in idx_multi:
    n_extra    = np.random.randint(1, 6)
    fecha_base = transacciones_base[idx][1]
    for _ in range(n_extra):
        nueva_fecha = fecha_base + timedelta(days=random.randint(30,500))
        if nueva_fecha > FECHA_FIN:
            nueva_fecha = FECHA_FIN
        transacciones_base.append((idx, nueva_fecha))

n_devol   = int(len(transacciones_base) * PCT_DEVOLUCION)
devol_set = set(np.random.choice(len(transacciones_base), n_devol, replace=False))
print(f'Total transacciones : {len(transacciones_base):,}')
print(f'Devoluciones        : {n_devol:,}')

Asignando transacciones...
Total transacciones : 5,058
Devoluciones        : 141


In [6]:
print('Construyendo dataset...')
registros      = []
num_factura    = 10000

for tx_idx, (cli_idx, fecha) in enumerate(transacciones_base):
    c             = df_clientes.iloc[cli_idx]
    es_devolucion = tx_idx in devol_set
    seccion  = np.random.choice(SEC_NAMES, p=SEC_PROBS)
    zona     = np.random.choice(ZONA_NAMES, p=ZONA_PROBS)
    bodega   = BODEGA_MAP[zona]
    linea    = np.random.choice(LINEA_NAMES, p=LINEA_PROBS)
    vendedor = random.choice(VENDEDORES)
    p_min,p_max,p_mean,p_std = PRECIO_PARAMS.get(seccion,(50,300,130,40))
    precio   = round(float(np.clip(np.random.normal(p_mean,p_std),p_min,p_max)),4)
    unidades = 1
    if es_devolucion:
        precio = -round(abs(precio),4)
        unidades = -1
    if seccion in CODIGOS_PRODUCTO:
        codigo = random.choice(CODIGOS_PRODUCTO[seccion])
        desc   = random.choice(DESCRIPCIONES[seccion])
    else:
        codigo = random.randint(5600000,5700000)
        desc   = f'PRODUCT {seccion} CRYSTAL'
    registros.append({
        'ANUAL':fecha.year,'MES':MESES_ES[fecha.month],'NUMERO':num_factura,
        'VENDEDOR':vendedor,'DESCRIPCION':desc,'CODIGO':codigo,'LINEA':linea,
        'SECCION':seccion,'RUC_CLIENTE':c['RUC_HASH'],'NOMBRE_CLIENTE':c['NOM_HASH'],
        'BODEGA':bodega,'ZONA':zona,'DOMICILIO':c['DOMICILIO'],
        'TELEFONO':c['TEL_HASH'],'MAIL':c['MAIL_HASH'],
        'FECHADOCUMENTO':fecha.strftime('%Y-%m-%d'),'UNIDADES':unidades,'VENTA':precio,
    })
    num_factura += 1

df_final = pd.DataFrame(registros).sort_values('FECHADOCUMENTO').reset_index(drop=True)
print(f'Dataset listo: {len(df_final):,} registros x {len(df_final.columns)} columnas')

Construyendo dataset...
Dataset listo: 5,058 registros x 18 columnas


In [7]:
print('='*55)
print('REPORTE DE CALIDAD')
print('='*55)
print(f'Registros totales  : {len(df_final):,}')
print(f'Clientes unicos    : {df_final["NOMBRE_CLIENTE"].nunique():,}')
print(f'Periodo            : {df_final["FECHADOCUMENTO"].min()} a {df_final["FECHADOCUMENTO"].max()}')
print(f'Ventas positivas   : {(df_final["VENTA"]>0).sum():,}')
print(f'Devoluciones       : {(df_final["VENTA"]<0).sum():,}')
print(f'Nulos totales      : {df_final.isnull().sum().sum()}')
print(f'Venta promedio     : ${df_final[df_final["VENTA"]>0]["VENTA"].mean():.2f}')
nulos_fecha = pd.to_datetime(df_final['FECHADOCUMENTO'],format='%Y-%m-%d',errors='coerce').isna().sum()
print(f'Nulos en fecha     : {nulos_fecha}  (OK si es 0)')
print()
print('Distribucion por seccion:')
print(df_final[df_final['VENTA']>0]['SECCION'].value_counts().to_string())
print('='*55)

REPORTE DE CALIDAD
Registros totales  : 5,058
Clientes unicos    : 3,000
Periodo            : 2023-01-01 a 2025-12-31
Ventas positivas   : 4,917
Devoluciones       : 141
Nulos totales      : 0
Venta promedio     : $145.85
Nulos en fecha     : 0  (OK si es 0)

Distribucion por seccion:
SECCION
COLLAR                            1352
ARETES                            1313
PULSERA                            801
ANILLO                             735
BOLIGRAFOS                         292
INDETERMINADO                      235
RELOJ Y SUS ACCESORIOS              98
FAMILIA 70-80                       42
LLAVEROS, MOBILE PHONE Y CHARM      27
HOME                                22


In [8]:
from google.colab import files

df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f'Archivo guardado: {OUTPUT_FILE}')
files.download(OUTPUT_FILE)
print('Descarga iniciada.')
print()
print('Proximos pasos:')
print('  1. Subir el CSV a Google Drive en /MyDrive/Proyecto_Swarovski/')
print("  2. En BLOQUE 1 del pipeline: FILE_DATASET = 'Dataset_Sintetico_Joyeria.csv'")
print('  3. Ejecutar todo el pipeline')

Archivo guardado: Dataset_Sintetico_Joyeria.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada.

Proximos pasos:
  1. Subir el CSV a Google Drive en /MyDrive/Proyecto_Swarovski/
  2. En BLOQUE 1 del pipeline: FILE_DATASET = 'Dataset_Sintetico_Joyeria.csv'
  3. Ejecutar todo el pipeline


---
**Nota:** Este dataset es completamente sintético. No contiene datos reales de ningún cliente ni empresa.  
**Autores:** Jhon Sánchez · Vinicio Betancourt — UDLA, 2026